## 5. Model Validation & Analyse des Erreurs (Test Set)

Jusqu'à présent, nous avons optimisé le modèle sur l'erreur absolue de la demande (MAE). Cependant, notre objectif final de Revenue Management est d'optimiser le revenu ($Prix \times Demande$). 

Une erreur de prévision de 3 billets sur un tarif à 15€ a un impact financier de 45€. La même erreur sur un tarif à 100€ a un impact de 300€. 
Nous allons donc évaluer notre modèle sur le jeu de test non pas seulement sur la justesse du volume (Demande), mais sur la **justesse du Revenu estimé**, en agrégeant les erreurs à différents niveaux stratégiques.

In [ ]:
# 1. Application du pipeline de Feature Engineering sur le set de TEST
test_fe = feature_engineering_pipeline(test)

# S'assurer que les variables catégorielles sont bien en string pour CatBoost
for col in cat_features:
    test_fe[col] = test_fe[col].astype(str)

# 2. Préparation des données de test
X_test = test_fe[features]
y_test = test_fe[target]

# 3. Prédictions
y_pred_test = model_cb.predict(X_test)
y_pred_test = np.maximum(0, y_pred_test)  # On tronque à 0 (une demande négative n'existe pas)

# 4. Création des métriques d'erreur dans le dataframe pour l'analyse
test_fe['predicted_demand'] = y_pred_test
test_fe['demand_error'] = test_fe['predicted_demand'] - test_fe['demand']
test_fe['demand_abs_error'] = test_fe['demand_error'].abs()

# Création des métriques orientées Business (REVENU)
test_fe['true_revenue'] = test_fe['demand'] * test_fe['price']
test_fe['predicted_revenue'] = test_fe['predicted_demand'] * test_fe['price']
test_fe['revenue_abs_error'] = (test_fe['predicted_revenue'] - test_fe['true_revenue']).abs()

print("--- PERFORMANCES GLOBALES SUR LE SET DE TEST ---")
print(f"MAE Demande (Volume) : {test_fe['demand_abs_error'].mean():.2f} billets par prédiction")
print(f"MAE Revenu (Impact Financier) : {test_fe['revenue_abs_error'].mean():.2f} € par prédiction")

### 5.1. Analyse des erreurs par Horizon de vente (`sale_day_x`)
Le comportement d'achat change drastiquement à la dernière minute. Le modèle est-il aussi performant à J-90 qu'à J-1 ?

In [ ]:
plt.figure(figsize=(14, 6))

# Calcul de l'erreur absolue moyenne par jour avant le départ
error_by_day_x = test_fe.groupby('sale_day_x').agg(
    mean_demand_error=('demand_abs_error', 'mean'),
    mean_revenue_error=('revenue_abs_error', 'mean')
).reset_index()

ax1 = sns.lineplot(data=error_by_day_x, x='sale_day_x', y='mean_demand_error', color='blue', label='Erreur Absolue Demande (Billets)')
ax1.set_ylabel('Erreur Demande (Billets)', color='blue')

# Ajout d'un axe secondaire pour le revenu
ax2 = ax1.twinx()
sns.lineplot(data=error_by_day_x, x='sale_day_x', y='mean_revenue_error', color='green', label='Erreur Absolue Revenu (€)', ax=ax2)
ax2.set_ylabel('Erreur Revenu (€)', color='green')

plt.title("Évolution de l'erreur de prédiction selon l'horizon de vente (Booking Curve)")
ax1.set_xlabel("Jours avant le départ (sale_day_x)")
ax1.invert_xaxis()
plt.show()

# INTERPRÉTATION :
# Généralement, l'erreur explose à J-10 car c'est là que la variance de la demande est la plus forte.
# Si l'erreur en revenu monte beaucoup plus vite que l'erreur en volume, cela signifie que notre modèle 
# se trompe particulièrement lorsque les billets sont les plus chers (dernière minute).

### 5.2. Analyse des erreurs par Trajet (OD level)
Certaines lignes sont structurellement plus difficiles à prédire (ex: lignes très touristiques avec beaucoup de volatilité vs lignes régulières de commuters).

In [ ]:
test_fe['OD'] = test_fe['origin_station_name'] + " - " + test_fe['destination_station_name']

# Calcul de l'erreur par ligne et sélection des 10 lignes générant le plus grand risque financier
error_by_od = test_fe.groupby('OD').agg(
    mean_revenue_error=('revenue_abs_error', 'mean'),
    mean_demand=('demand', 'mean'),
    prediction_count=('demand', 'count')
).sort_values(by='mean_revenue_error', ascending=False)

# On filtre les OD marginaux (moins de 50 prédictions) pour se concentrer sur les flux principaux
worst_ods = error_by_od[error_by_od['prediction_count'] > 50].head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=worst_ods['mean_revenue_error'], y=worst_ods.index, palette='Reds_r')
plt.title("Top 10 des Trajets (OD) avec la plus forte erreur financière moyenne")
plt.xlabel("Erreur Absolue Moyenne sur le Revenu (€)")
plt.ylabel("Origine - Destination")
plt.tight_layout()
plt.show()

# INTERPRÉTATION :
# Ces trajets nécessitent probablement des features dédiées (ex: événements locaux, météo sur la destination touristique)
# ou un modèle entraîné spécifiquement pour eux s'ils ont un comportement atypique.

### 5.3. Analyse de l'erreur par Paliers de Prix (Sensibilité Tarifaire)
Puisque le but est d'optimiser le revenu, nous devons vérifier si notre modèle sous-performe lorsque nous proposons des billets à des tarifs élevés. Un modèle qui prédit bien les promos mais mal les billets plein tarif est dangereux pour l'entreprise.

In [ ]:
# Création de 4 catégories de prix (Quartiles) pour l'analyse
test_fe['price_tier'] = pd.qcut(test_fe['price'], q=4, labels=['1. Low Price', '2. Medium-Low', '3. Medium-High', '4. Premium Price'])

error_by_price = test_fe.groupby('price_tier').agg(
    bias_demand=('demand_error', 'mean'),       # Biais : est-ce qu'on surestime (+) ou sous-estime (-) ?
    mae_revenue=('revenue_abs_error', 'mean')   # Ampleur de l'erreur financière
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Graphique 1 : Le Biais (Surestimation vs Sous-estimation)
sns.barplot(data=error_by_price, x='price_tier', y='bias_demand', ax=axes[0], palette='coolwarm')
axes[0].set_title("Biais de Prédiction par Niveau de Prix")
axes[0].set_ylabel("Biais Moyen (Billets)\n(>0 : Le modèle surestime | <0 : Il sous-estime)")
axes[0].axhline(0, color='black', linestyle='--')

# Graphique 2 : L'impact financier direct
sns.barplot(data=error_by_price, x='price_tier', y='mae_revenue', ax=axes[1], palette='Oranges')
axes[1].set_title("Impact Financier de l'erreur (MAE Revenu) par Niveau de Prix")
axes[1].set_ylabel("Erreur Absolue Moyenne du Revenu (€)")

plt.tight_layout()
plt.show()

# INTERPRÉTATION :
# Le graphique de gauche (Biais) est fondamental. S'il est très négatif pour la catégorie 'Premium',
# cela signifie que notre algorithme pense que personne n'achètera à ce prix-là, alors que les gens achètent.
# Cela pousserait le Yield Manager à baisser ses prix à tort, détruisant ainsi la rentabilité du train.